# Notebook 1: Data Collection (Maximum Volume v3)
## Songwriter Style Analysis - Maximum Volume Data Collection

**Objective**: Collect 150+ songs per songwriter for 60-70%+ accuracy

**Target Songwriters**: 6 top songwriters (focused dataset with expanded artist lists)

**APIs Used**:
- Genius API: Lyrics and songwriter credits
- Last.fm API: Popularity metrics and tags

**Collection Strategy**:
- 24 artists per songwriter (expanded from 12)
- Max 30 songs per artist (increased from 20)
- Target: 150+ songs per songwriter

**Expected Output**: songs_data_final.csv with 900+ songs
**Target per songwriter**: 150+ songs minimum

## Step 1: Import Libraries and Setup

In [1]:
# Import essential libraries
import lyricsgenius as lg
import pylast
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully
Pandas version: 2.2.3
NumPy version: 2.2.2


## Step 2: Configure API Credentials

In [2]:
# Import API credentials
from config import GENIUS_TOKEN, LASTFM_API_KEY, LASTFM_API_SECRET

# Initialize Genius API
genius = lg.Genius(
    GENIUS_TOKEN,
    skip_non_songs=True,
    remove_section_headers=True,
    verbose=False,
    timeout=15,
    retries=3
)

# Initialize Last.fm API
lastfm_network = pylast.LastFMNetwork(
    api_key=LASTFM_API_KEY,
    api_secret=LASTFM_API_SECRET
)

print("API clients initialized successfully")
print(f"Genius Token: {GENIUS_TOKEN[:20]}...")
print(f"Last.fm API Key: {LASTFM_API_KEY[:20]}...")

API clients initialized successfully
Genius Token: z2UefZTuteHUDbYCjXil...
Last.fm API Key: 595d78b760f83e24e7cb...


## Step 3: Test API Connections

In [3]:
# Test Genius API
print("Testing Genius API connection...")
print("="*60)

try:
    test_song = genius.search_song("Blinding Lights", "The Weeknd")
    if test_song:
        print("[SUCCESS] Genius API connection working")
        print(f"Song: {test_song.title}")
        print(f"Artist: {test_song.artist}")
        print(f"Lyrics length: {len(test_song.lyrics)} characters")
        
        # Check for writer credits
        if hasattr(test_song, '_body') and 'writer_artists' in test_song._body:
            writers = [w['name'] for w in test_song._body['writer_artists']]
            print(f"Writers: {', '.join(writers)}")
    else:
        print("[ERROR] Song not found")
except Exception as e:
    print(f"[ERROR] {str(e)}")

print("\n" + "="*60)

# Test Last.fm API
print("Testing Last.fm API connection...")
print("="*60)

try:
    track = lastfm_network.get_track("The Weeknd", "Blinding Lights")
    print("[SUCCESS] Last.fm API connection working")
    print(f"Track: {track.get_name()}")
    print(f"Artist: {track.get_artist()}")
    print(f"Playcount: {track.get_playcount():,}")
    print(f"Listeners: {track.get_listener_count():,}")
except Exception as e:
    print(f"[ERROR] {str(e)}")

print("\n" + "="*60)
print("API connection tests complete")

Testing Genius API connection...
[SUCCESS] Genius API connection working
Song: Blinding Lights
Artist: The Weeknd
Lyrics length: 1266 characters
Writers: The Weeknd, Oscar Holter, Max Martin, Belly, DaHeala

Testing Last.fm API connection...
[SUCCESS] Last.fm API connection working
Track: Blinding Lights
Artist: The Weeknd
[SUCCESS] Genius API connection working
Song: Blinding Lights
Artist: The Weeknd
Lyrics length: 1266 characters
Writers: The Weeknd, Oscar Holter, Max Martin, Belly, DaHeala

Testing Last.fm API connection...
[SUCCESS] Last.fm API connection working
Track: Blinding Lights
Artist: The Weeknd
Playcount: 36,481,023
Listeners: 2,221,943

API connection tests complete
Playcount: 36,481,023
Listeners: 2,221,943

API connection tests complete


## Step 4: Define Target Songwriters (Expanded List)

In [4]:
# FOCUSED songwriter list - Top 6 songwriters with MASSIVELY EXPANDED artist lists
# Target: 150+ songs per songwriter for optimal ML accuracy
target_writers = {
    'Jack Antonoff': {
        'known_for': '80s-inspired production, indie-pop, emotional storytelling',
        'notable_artists': ['Taylor Swift', 'Lorde', 'Lana Del Rey', 'Bleachers', 
                           'St. Vincent', 'Carly Rae Jepsen', 'The Chicks', 'Clairo',
                           'Pink', 'Troye Sivan', 'Sara Bareilles', 'Griff',
                           'MUNA', 'girl in red', 'Chloe Moriondo', 'Kevin Abstract',
                           'Charli XCX', 'FKA twigs', 'Diana Gordon', 'The 1975',
                           'Scary Pockets', 'Tegan and Sara', 'Lykke Li', 'Florence + The Machine']
    },
    'Max Martin': {
        'known_for': 'Pop perfection, catchy hooks, radio-friendly hits',
        'notable_artists': ['Taylor Swift', 'The Weeknd', 'Ariana Grande', 'Katy Perry', 
                           'Maroon 5', 'P!nk', 'Backstreet Boys', 'Britney Spears',
                           'Demi Lovato', 'Bon Jovi', 'Usher', 'Ellie Goulding',
                           'Kelly Clarkson', 'Jessie J', 'Tove Lo', 'Carly Rae Jepsen',
                           '*NSYNC', 'Céline Dion', 'Robyn', 'Adam Lambert',
                           'Avicii', 'Nicki Minaj', 'Justin Timberlake', 'Anne-Marie']
    },
    'Dr. Luke': {
        'known_for': 'Electronic pop, club bangers, provocative themes',
        'notable_artists': ['Katy Perry', 'Kesha', 'Doja Cat', 'Kim Petras', 
                           'P!nk', 'Miley Cyrus', 'Avril Lavigne', 'Taio Cruz',
                           'Flo Rida', 'B.o.B', 'Nicki Minaj', 'Pitbull',
                           'Britney Spears', 'Kelly Clarkson', 'Jessie J', 'Juicy J',
                           'Becky G', 'Saweetie', 'Tyga', 'R. City',
                           'Wiz Khalifa', 'The Veronicas', 'JoJo', 'Ciara']
    },
    'Ryan Tedder': {
        'known_for': 'Anthemic choruses, piano-driven, emotional crescendos',
        'notable_artists': ['Beyoncé', 'Adele', 'Taylor Swift', 'OneRepublic', 
                           'Leona Lewis', 'Ed Sheeran', 'Jonas Brothers', 'U2',
                           'Colbie Caillat', 'James Blunt', 'Ariana Grande', 'Camila Cabello',
                           'Demi Lovato', 'Maroon 5', 'Jennifer Lopez', 'Kelly Clarkson',
                           'Shawn Mendes', 'Ella Henderson', 'Hailee Steinfeld', 'Zedd',
                           'Galantis', 'Kygo', 'Lost Frequencies', 'MØ']
    },
    'Sia Furler': {
        'known_for': 'Powerful vocals, emotional depth, EDM collaboration',
        'notable_artists': ['Rihanna', 'Beyoncé', 'David Guetta', 'Sia', 
                           'Flo Rida', 'Eminem', 'Christina Aguilera', 'Katy Perry',
                           'Britney Spears', 'Céline Dion', 'Rita Ora', 'Zayn',
                           'Ne-Yo', 'Kylie Minogue', 'Jessie J', 'Lea Michele',
                           'Ariana Grande', 'Demi Lovato', 'Miguel', 'J Balvin',
                           'Diplo', 'LSD', 'Labrinth', 'Adele']
    },
    'Stargate': {
        'known_for': 'R&B grooves, pop-soul fusion, smooth production',
        'notable_artists': ['Rihanna', 'Beyoncé', 'Ne-Yo', 'Katy Perry', 
                           'Sam Smith', 'Wiz Khalifa', 'Coldplay', 'Lionel Richie',
                           'Shakira', 'Mariah Carey', 'Jennifer Lopez', 'Usher',
                           'Chris Brown', 'Janet Jackson', 'Mary J. Blige', 'Normani',
                           'Halsey', 'Camila Cabello', 'Cardi B', 'SZA',
                           'Jhené Aiko', 'Tinashe', 'Pia Mia', 'Trey Songz']
    }
}

print("MAXIMUM VOLUME Data Collection Strategy")
print("="*60)
print("FOCUSED APPROACH: Top 6 songwriters with MASSIVELY EXPANDED artist lists")
print("Target: 150+ songs per songwriter (vs previous 80-100)")
print("Why: Maximize training data for best possible accuracy (target 60-70%)")
print("\n" + "="*60)

for writer, info in target_writers.items():
    print(f"\n{writer}")
    print(f"  Style: {info['known_for']}")
    print(f"  Artists ({len(info['notable_artists'])}): {', '.join(info['notable_artists'][:5])}...")

print("\n" + "="*60)
print(f"Total Songwriters: {len(target_writers)}")
print(f"Artists per songwriter: 24")
print(f"Target per songwriter: 150+ songs")
print(f"Expected Total: 900+ songs")
print(f"Expected Accuracy after training: 60-70%")

MAXIMUM VOLUME Data Collection Strategy
FOCUSED APPROACH: Top 6 songwriters with MASSIVELY EXPANDED artist lists
Target: 150+ songs per songwriter (vs previous 80-100)
Why: Maximize training data for best possible accuracy (target 60-70%)


Jack Antonoff
  Style: 80s-inspired production, indie-pop, emotional storytelling
  Artists (24): Taylor Swift, Lorde, Lana Del Rey, Bleachers, St. Vincent...

Max Martin
  Style: Pop perfection, catchy hooks, radio-friendly hits
  Artists (24): Taylor Swift, The Weeknd, Ariana Grande, Katy Perry, Maroon 5...

Dr. Luke
  Style: Electronic pop, club bangers, provocative themes
  Artists (24): Katy Perry, Kesha, Doja Cat, Kim Petras, P!nk...

Ryan Tedder
  Style: Anthemic choruses, piano-driven, emotional crescendos
  Artists (24): Beyoncé, Adele, Taylor Swift, OneRepublic, Leona Lewis...

Sia Furler
  Style: Powerful vocals, emotional depth, EDM collaboration
  Artists (24): Rihanna, Beyoncé, David Guetta, Sia, Flo Rida...

Stargate
  Style: R&B groo

### Clean Up Old Checkpoint Files (if any exist from previous runs)

In [5]:
# Verify we have exactly 6 songwriters
print("SONGWRITER CONFIGURATION VERIFICATION")
print("="*60)
print(f"Number of songwriters configured: {len(target_writers)}")
print(f"Songwriters: {list(target_writers.keys())}")

if len(target_writers) != 6:
    print(f"\n⚠️  WARNING: Expected 6 songwriters, but found {len(target_writers)}!")
else:
    print("\n✅ Configuration verified: Exactly 6 songwriters!")

# Clean up old checkpoint files from previous runs (if any)
import glob

print("\n" + "="*60)
print("CLEANING UP OLD CHECKPOINT FILES")
print("="*60)

if os.path.exists('data'):
    # Get all checkpoint files
    checkpoint_files = glob.glob('data/checkpoint_*.csv')
    
    # Get list of valid checkpoint files for current songwriters
    valid_checkpoints = set()
    for songwriter_name in target_writers.keys():
        valid_file = f"data/checkpoint_{songwriter_name.replace(' ', '_')}.csv"
        valid_checkpoints.add(valid_file)
    
    # Remove old checkpoint files
    removed_count = 0
    for checkpoint_file in checkpoint_files:
        if checkpoint_file not in valid_checkpoints:
            try:
                os.remove(checkpoint_file)
                songwriter_name = checkpoint_file.replace('data/checkpoint_', '').replace('.csv', '').replace('_', ' ')
                print(f"[REMOVED] Old checkpoint for: {songwriter_name}")
                removed_count += 1
            except Exception as e:
                print(f"[ERROR] Could not remove {checkpoint_file}: {str(e)}")
    
    if removed_count > 0:
        print(f"\n✅ Cleaned up {removed_count} old checkpoint file(s)")
    else:
        print("\n✅ No old checkpoint files to remove")
else:
    print("Data directory does not exist yet - will be created during collection")

print("="*60)

SONGWRITER CONFIGURATION VERIFICATION
Number of songwriters configured: 6
Songwriters: ['Jack Antonoff', 'Max Martin', 'Dr. Luke', 'Ryan Tedder', 'Sia Furler', 'Stargate']

✅ Configuration verified: Exactly 6 songwriters!

CLEANING UP OLD CHECKPOINT FILES
[REMOVED] Old checkpoint for: data\checkpoint Benny Blanco
[REMOVED] Old checkpoint for: data\checkpoint Diplo
[REMOVED] Old checkpoint for: data\checkpoint Dr. Luke
[REMOVED] Old checkpoint for: data\checkpoint Greg Kurstin
[REMOVED] Old checkpoint for: data\checkpoint Jack Antonoff
[REMOVED] Old checkpoint for: data\checkpoint Julia Michaels
[REMOVED] Old checkpoint for: data\checkpoint Max Martin
[REMOVED] Old checkpoint for: data\checkpoint Ryan Tedder
[REMOVED] Old checkpoint for: data\checkpoint Shellback
[REMOVED] Old checkpoint for: data\checkpoint Sia Furler
[REMOVED] Old checkpoint for: data\checkpoint Stargate
[REMOVED] Old checkpoint for: data\checkpoint The-Dream

✅ Cleaned up 12 old checkpoint file(s)


## Step 5: Helper Functions for Data Collection

In [6]:
def extract_song_data(song, target_songwriter):
    """
    Extract comprehensive data from a Genius song object
    
    Args:
        song: Genius song object
        target_songwriter: Name of the songwriter we're collecting for
    
    Returns:
        dict: Song data or None if invalid
    """
    if not song or not hasattr(song, 'lyrics'):
        return None
    
    # Skip songs without lyrics or very short lyrics
    if not song.lyrics or len(song.lyrics.strip()) < 100:
        return None
    
    song_data = {
        'title': song.title,
        'artist': song.artist,
        'lyrics': song.lyrics,
        'target_songwriter': target_songwriter,
        'writers': 'Unknown',
        'producers': 'Unknown',
        'release_date': None,
        'url': song.url if hasattr(song, 'url') else None,
        'pageviews': 0,
        'lastfm_playcount': 0,
        'lastfm_listeners': 0,
        'lastfm_tags': ''
    }
    
    # Extract metadata from Genius
    if hasattr(song, '_body'):
        metadata = song._body
        
        # Get writers
        if 'writer_artists' in metadata:
            writers = [w['name'] for w in metadata['writer_artists']]
            song_data['writers'] = ', '.join(writers)
        
        # Get producers
        if 'producer_artists' in metadata:
            producers = [p['name'] for p in metadata['producer_artists']]
            song_data['producers'] = ', '.join(producers)
        
        # Get release date
        if 'release_date' in metadata and metadata['release_date']:
            song_data['release_date'] = metadata['release_date']
        
        # Get pageviews
        if 'stats' in metadata and 'pageviews' in metadata['stats']:
            song_data['pageviews'] = metadata['stats']['pageviews']
    
    return song_data


def get_lastfm_data(artist_name, song_title):
    """
    Get popularity metrics from Last.fm
    
    Args:
        artist_name: Artist name
        song_title: Song title
    
    Returns:
        dict: Last.fm metrics
    """
    lastfm_data = {
        'playcount': 0,
        'listeners': 0,
        'tags': ''
    }
    
    try:
        track = lastfm_network.get_track(artist_name, song_title)
        lastfm_data['playcount'] = track.get_playcount()
        lastfm_data['listeners'] = track.get_listener_count()
        
        # Get tags
        tags = track.get_top_tags(limit=3)
        lastfm_data['tags'] = ', '.join([tag.item.get_name() for tag in tags])
    except Exception as e:
        # Log Last.fm failures for tracking data quality
        print(f"      [Last.fm ERROR] {artist_name} - {song_title}: {str(e)[:50]}")
    
    return lastfm_data


def save_checkpoint(songwriter_name, songs_data):
    """
    Save checkpoint for a songwriter's collected data
    
    Args:
        songwriter_name: Name of songwriter
        songs_data: List of song dictionaries
    """
    if not os.path.exists('data'):
        os.makedirs('data')
    
    checkpoint_file = f"data/checkpoint_{songwriter_name.replace(' ', '_')}.csv"
    df = pd.DataFrame(songs_data)
    df.to_csv(checkpoint_file, index=False)
    print(f"  [CHECKPOINT] Saved {len(songs_data)} songs to {checkpoint_file}")


print("Helper functions defined successfully")
print("  - extract_song_data(): Extract metadata from Genius")
print("  - get_lastfm_data(): Get popularity metrics (with error logging)")
print("  - save_checkpoint(): Save progress during collection")


Helper functions defined successfully
  - extract_song_data(): Extract metadata from Genius
  - get_lastfm_data(): Get popularity metrics (with error logging)
  - save_checkpoint(): Save progress during collection


## Step 6: MAXIMUM VOLUME Data Collection (150+ Songs Per Songwriter)

In [7]:
# MAXIMUM VOLUME data collection - targeting 150+ songs per songwriter
print("STARTING MAXIMUM VOLUME DATA COLLECTION")
print("="*60)
print(f"Target: {len(target_writers)} songwriters (FOCUSED APPROACH)")
print(f"Goal: 150+ songs per songwriter (INCREASED from 80-100)")
print(f"Expected total: 900+ songs")
print(f"Why: More data = better accuracy (targeting 60-70% accuracy)")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

# Storage
songwriter_stats = {}

# Collection parameters - MAXIMUM VOLUME SETTINGS
SONGS_PER_SONGWRITER_TARGET = 150  # Increased from 100 to 150
MAX_SONGS_PER_ARTIST = 30          # Increased from 20 to 30
RATE_LIMIT_DELAY = 2.0             # Increased from 1.5 to 2.0 seconds between songs
ARTIST_DELAY = 4.0                 # Increased from 3.0 to 4.0 seconds between artists
RATE_LIMIT_WAIT = 60               # Seconds to wait if rate limited

# Load existing checkpoints to resume if interrupted
for songwriter_name in target_writers.keys():
    checkpoint_file = f"data/checkpoint_{songwriter_name.replace(' ', '_')}.csv"
    if os.path.exists(checkpoint_file):
        existing_df = pd.read_csv(checkpoint_file)
        songwriter_stats[songwriter_name] = len(existing_df)
        print(f"[RESUME] Loaded {len(existing_df)} existing songs for {songwriter_name}")

# Collect songs for each songwriter
for songwriter_name, writer_info in target_writers.items():
    print(f"\n{'='*60}")
    print(f"COLLECTING FOR: {songwriter_name}")
    print(f"{'='*60}")
    print(f"Style: {writer_info['known_for']}")
    print(f"Target artists: {len(writer_info['notable_artists'])}")
    
    # Get existing songs count
    existing_count = songwriter_stats.get(songwriter_name, 0)
    print(f"Existing songs: {existing_count}")
    print(f"Target: {SONGS_PER_SONGWRITER_TARGET} songs")
    print(f"Need to collect: {max(0, SONGS_PER_SONGWRITER_TARGET - existing_count)} more songs")
    
    if existing_count >= SONGS_PER_SONGWRITER_TARGET:
        print(f"[SKIP] Already have enough songs for {songwriter_name}")
        continue
    
    songwriter_songs = []
    
    # Load existing songs if checkpoint exists
    checkpoint_file = f"data/checkpoint_{songwriter_name.replace(' ', '_')}.csv"
    if os.path.exists(checkpoint_file):
        existing_df = pd.read_csv(checkpoint_file)
        songwriter_songs = existing_df.to_dict('records')
    
    songs_collected = len(songwriter_songs)
    
    # Search through each artist
    for artist_idx, artist_name in enumerate(writer_info['notable_artists'], 1):
        if songs_collected >= SONGS_PER_SONGWRITER_TARGET:
            print(f"\n[TARGET REACHED] Collected {songs_collected} songs for {songwriter_name}")
            break
        
        print(f"\n[{artist_idx}/{len(writer_info['notable_artists'])}] Searching: {artist_name}")
        
        try:
            # Search for artist on Genius
            artist = genius.search_artist(artist_name, max_songs=MAX_SONGS_PER_ARTIST)
            
            if not artist:
                print(f"  [NOT FOUND] Artist '{artist_name}' not found")
                time.sleep(ARTIST_DELAY)
                continue
            
            print(f"  Found artist: {artist.name} ({len(artist.songs)} songs)")
            
            # Process each song
            for song_idx, song in enumerate(artist.songs, 1):
                if songs_collected >= SONGS_PER_SONGWRITER_TARGET:
                    break
                
                try:
                    # Extract song data
                    song_data = extract_song_data(song, songwriter_name)
                    
                    if not song_data:
                        continue
                    
                    # STRICT songwriter credit validation - exact name match only
                    writers_list = song_data['writers'].lower()
                    songwriter_name_lower = songwriter_name.lower()
                    
                    # For exact matching, check if the full songwriter name appears in the writers list
                    # surrounded by word boundaries (commas, start/end of string)
                    is_credited = False
                    
                    # Split writers by comma and check each one
                    individual_writers = [w.strip() for w in writers_list.split(',')]
                    for writer in individual_writers:
                        # Exact match (ignoring case)
                        if songwriter_name_lower == writer:
                            is_credited = True
                            break
                        # Also check if songwriter name is contained as a full substring
                        # This handles cases like "Max Martin" in "Martin Sandberg (Max Martin)"
                        if songwriter_name_lower in writer and len(songwriter_name_lower) > 5:
                            is_credited = True
                            break
                    
                    if not is_credited:
                        continue
                    
                    # Check for duplicates
                    is_duplicate = any(
                        s['title'].lower() == song_data['title'].lower() and 
                        s['artist'].lower() == song_data['artist'].lower()
                        for s in songwriter_songs
                    )
                    
                    if is_duplicate:
                        continue
                    
                    # Get Last.fm data
                    lastfm_data = get_lastfm_data(song_data['artist'], song_data['title'])
                    song_data['lastfm_playcount'] = lastfm_data['playcount']
                    song_data['lastfm_listeners'] = lastfm_data['listeners']
                    song_data['lastfm_tags'] = lastfm_data['tags']
                    
                    # Add song
                    songwriter_songs.append(song_data)
                    songs_collected += 1
                    
                    print(f"    [{songs_collected}/{SONGS_PER_SONGWRITER_TARGET}] Added: {song_data['title']} by {song_data['artist']}")
                    
                    # Rate limiting
                    time.sleep(RATE_LIMIT_DELAY)
                    
                except Exception as e:
                    print(f"    [ERROR] Processing song: {str(e)[:50]}")
                    continue
            
            # Delay between artists
            time.sleep(ARTIST_DELAY)
            
        except Exception as e:
            error_msg = str(e)
            if '429' in error_msg or 'rate limit' in error_msg.lower():
                print(f"  [RATE LIMIT] Waiting {RATE_LIMIT_WAIT} seconds...")
                time.sleep(RATE_LIMIT_WAIT)
            else:
                print(f"  [ERROR] Searching artist: {error_msg[:100]}")
            continue
    
    # Save checkpoint after each songwriter
    if songwriter_songs:
        save_checkpoint(songwriter_name, songwriter_songs)
        songwriter_stats[songwriter_name] = len(songwriter_songs)
    
    print(f"\n[COMPLETE] {songwriter_name}: {len(songwriter_songs)} songs collected")

# Combine all songs from checkpoints (single loading, no duplicates)
print(f"\n{'='*60}")
print("COLLECTION SUMMARY")
print(f"{'='*60}")

all_songs_data = []
for songwriter_name in target_writers.keys():
    checkpoint_file = f"data/checkpoint_{songwriter_name.replace(' ', '_')}.csv"
    if os.path.exists(checkpoint_file):
        df = pd.read_csv(checkpoint_file)
        all_songs_data.extend(df.to_dict('records'))
        print(f"  {songwriter_name}: {len(df)} songs")

print(f"\nTotal songs collected: {len(all_songs_data)}")
print(f"Average per songwriter: {len(all_songs_data) / len(target_writers):.1f}")
print(f"Collection completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")


STARTING MAXIMUM VOLUME DATA COLLECTION
Target: 6 songwriters (FOCUSED APPROACH)
Goal: 150+ songs per songwriter (INCREASED from 80-100)
Expected total: 900+ songs
Why: More data = better accuracy (targeting 60-70% accuracy)
Started: 2025-11-11 11:42:13

COLLECTING FOR: Jack Antonoff
Style: 80s-inspired production, indie-pop, emotional storytelling
Target artists: 24
Existing songs: 0
Target: 150 songs
Need to collect: 150 more songs

[1/24] Searching: Taylor Swift
  Found artist: Taylor Swift (30 songs)
  Found artist: Taylor Swift (30 songs)
    [1/150] Added: Fortnight by Taylor Swift
    [1/150] Added: Fortnight by Taylor Swift
    [2/150] Added: The Tortured Poets Department by Taylor Swift
    [2/150] Added: The Tortured Poets Department by Taylor Swift
    [3/150] Added: Down Bad by Taylor Swift
    [3/150] Added: Down Bad by Taylor Swift
    [4/150] Added: Is It Over Now? (Taylor’s Version) [From the Vault] by Taylor Swift
    [4/150] Added: Is It Over Now? (Taylor’s Version) [

## Step 7: Save Final Dataset

In [8]:
# Create DataFrame from collected data
df_songs = pd.DataFrame(all_songs_data)

print("Dataset Overview")
print("="*60)
print(f"Total Songs: {len(df_songs)}")
print(f"Total Columns: {len(df_songs.columns)}")
print(f"\nColumns: {list(df_songs.columns)}")

print("\nSongs per Songwriter:")
print(df_songs['target_songwriter'].value_counts())

print("\nDataset Statistics:")
print(f"  Average lyrics length: {df_songs['lyrics'].str.len().mean():.0f} characters")
print(f"  Median lyrics length: {df_songs['lyrics'].str.len().median():.0f} characters")
print(f"  Missing values: {df_songs.isnull().sum().sum()}")

# Save to CSV
output_file = 'data/songs_data_final.csv'
df_songs.to_csv(output_file, index=False)
print(f"\n[SAVED] Dataset saved to {output_file}")

# Also save as JSON backup
json_file = 'data/songs_data_final.json'
df_songs.to_json(json_file, orient='records', indent=2)
print(f"[SAVED] JSON backup saved to {json_file}")

# Save metadata
metadata = {
    'collection_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_songs': len(df_songs),
    'songwriters': list(songwriter_stats.keys()),
    'songs_per_songwriter': songwriter_stats
}

import json
metadata_file = 'data/collection_metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"[SAVED] Metadata saved to {metadata_file}")

print("\n" + "="*60)
print("DATA COLLECTION NOTEBOOK COMPLETE")
print("="*60)
print(f"\nNext Step: Run 02_preprocessing.ipynb")

Dataset Overview
Total Songs: 413
Total Columns: 12

Columns: ['title', 'artist', 'lyrics', 'target_songwriter', 'writers', 'producers', 'release_date', 'url', 'pageviews', 'lastfm_playcount', 'lastfm_listeners', 'lastfm_tags']

Songs per Songwriter:
target_songwriter
Dr. Luke         129
Max Martin       110
Jack Antonoff    100
Ryan Tedder       64
Stargate          10
Name: count, dtype: int64

Dataset Statistics:
  Average lyrics length: 1950 characters
  Median lyrics length: 1856 characters
  Missing values: 298

[SAVED] Dataset saved to data/songs_data_final.csv
[SAVED] JSON backup saved to data/songs_data_final.json
[SAVED] Metadata saved to data/collection_metadata.json

DATA COLLECTION NOTEBOOK COMPLETE

Next Step: Run 02_preprocessing.ipynb


## Step 8: Data Quality Validation

In [9]:
print("DATA QUALITY VALIDATION")
print("="*60)

# 1. Class Balance Check
print("\n1. CLASS BALANCE:")
print("-"*60)
class_counts = df_songs['target_songwriter'].value_counts()
print(class_counts)
print(f"\nMinimum songs: {class_counts.min()}")
print(f"Maximum songs: {class_counts.max()}")
print(f"Balance ratio: {class_counts.min() / class_counts.max():.2%}")

if class_counts.min() / class_counts.max() < 0.7:
    print("⚠️  WARNING: Class imbalance detected! Consider collecting more data for underrepresented songwriters.")
else:
    print("✅ Good class balance!")

# 2. Credit Verification Check
print("\n\n2. CREDIT VERIFICATION:")
print("-"*60)

def verify_strict_credit(row):
    """Verify songwriter is actually credited in the writers list"""
    writers_list = str(row['writers']).lower()
    songwriter_name_lower = row['target_songwriter'].lower()
    
    # Split writers by comma and check each one
    individual_writers = [w.strip() for w in writers_list.split(',')]
    for writer in individual_writers:
        if songwriter_name_lower == writer or songwriter_name_lower in writer:
            return True
    return False

df_songs['credit_verified'] = df_songs.apply(verify_strict_credit, axis=1)
verified_count = df_songs['credit_verified'].sum()
total_count = len(df_songs)

print(f"Verified credits: {verified_count} / {total_count} ({verified_count/total_count:.1%})")

if verified_count < total_count:
    unverified = df_songs[~df_songs['credit_verified']]
    print(f"\n⚠️  WARNING: {len(unverified)} songs without verified credits!")
    print("\nSample unverified songs:")
    print(unverified[['title', 'artist', 'target_songwriter', 'writers']].head(5))
else:
    print("✅ All songs have verified songwriter credits!")

# 3. Check for Overlapping Songs (same song attributed to multiple songwriters)
print("\n\n3. OVERLAPPING SONGS CHECK:")
print("-"*60)

from collections import defaultdict
song_to_writers = defaultdict(list)

for _, row in df_songs.iterrows():
    song_key = (row['title'].lower(), row['artist'].lower())
    song_to_writers[song_key].append(row['target_songwriter'])

overlapping_songs = {k: v for k, v in song_to_writers.items() if len(set(v)) > 1}

print(f"Songs appearing with multiple songwriters: {len(overlapping_songs)}")

if overlapping_songs:
    print("\n⚠️  WARNING: Data leakage detected! Same songs attributed to multiple songwriters.")
    print("\nSample overlapping songs:")
    for i, (song_key, writers) in enumerate(list(overlapping_songs.items())[:5]):
        print(f"  {i+1}. '{song_key[0]}' by {song_key[1]}")
        print(f"     → Attributed to: {', '.join(set(writers))}")
else:
    print("✅ No overlapping songs detected!")

# 4. Check for Artist Overlap Between Songwriters
print("\n\n4. ARTIST OVERLAP ANALYSIS:")
print("-"*60)

artist_to_writers = defaultdict(list)
for _, row in df_songs.iterrows():
    artist_to_writers[row['artist']].append(row['target_songwriter'])

overlapping_artists = {k: list(set(v)) for k, v in artist_to_writers.items() if len(set(v)) > 1}

print(f"Artists appearing with multiple songwriters: {len(overlapping_artists)}")

if overlapping_artists:
    print("\nℹ️  This is normal - many artists work with multiple songwriters!")
    print("\nTop overlapping artists:")
    sorted_artists = sorted(overlapping_artists.items(), key=lambda x: len(x[1]), reverse=True)
    for i, (artist, writers) in enumerate(sorted_artists[:10]):
        song_count = len([s for s in df_songs[df_songs['artist'] == artist]['target_songwriter']])
        print(f"  {i+1}. {artist} ({song_count} songs)")
        print(f"     → Songwriters: {', '.join(writers)}")

# 5. Data Completeness Check
print("\n\n5. DATA COMPLETENESS:")
print("-"*60)

print("Missing values per column:")
missing = df_songs.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values!")

print(f"\nAverage lyrics length: {df_songs['lyrics'].str.len().mean():.0f} characters")
print(f"Minimum lyrics length: {df_songs['lyrics'].str.len().min():.0f} characters")

short_lyrics = df_songs[df_songs['lyrics'].str.len() < 500]
if len(short_lyrics) > 0:
    print(f"\n⚠️  {len(short_lyrics)} songs with very short lyrics (< 500 characters)")
else:
    print("✅ All songs have adequate lyrics length!")

# 6. Metadata Quality
print("\n\n6. METADATA QUALITY:")
print("-"*60)

unknown_writers = df_songs[df_songs['writers'] == 'Unknown']
print(f"Songs with unknown writers: {len(unknown_writers)} ({len(unknown_writers)/len(df_songs):.1%})")

has_lastfm = df_songs[df_songs['lastfm_playcount'] > 0]
print(f"Songs with Last.fm data: {len(has_lastfm)} ({len(has_lastfm)/len(df_songs):.1%})")

print("\n" + "="*60)
print("VALIDATION COMPLETE")
print("="*60)

DATA QUALITY VALIDATION

1. CLASS BALANCE:
------------------------------------------------------------
target_songwriter
Dr. Luke         129
Max Martin       110
Jack Antonoff    100
Ryan Tedder       64
Stargate          10
Name: count, dtype: int64

Minimum songs: 10
Maximum songs: 129
Balance ratio: 7.75%
⚠️  WARNING: Class imbalance detected! Consider collecting more data for underrepresented songwriters.


2. CREDIT VERIFICATION:
------------------------------------------------------------
Verified credits: 413 / 413 (100.0%)
✅ All songs have verified songwriter credits!


3. OVERLAPPING SONGS CHECK:
------------------------------------------------------------
Songs appearing with multiple songwriters: 22

⚠️  WARNING: Data leakage detected! Same songs attributed to multiple songwriters.

Sample overlapping songs:
  1. 'california gurls' by katy perry
     → Attributed to: Max Martin, Dr. Luke
  2. 'dark horse' by katy perry
     → Attributed to: Max Martin, Dr. Luke
  3. 'the o

In [10]:
# Display sample of collected data
print("Sample of Collected Data:")
print("="*60)
df_songs[['title', 'artist', 'target_songwriter', 'writers', 'lastfm_playcount']].head(10)

Sample of Collected Data:


,title,artist,target_songwriter,writers,lastfm_playcount
0,Fortnight,Taylor Swift,Jack Antonoff,"Taylor Swift, Jack Antonoff, Post Malone",200805
1,The Tortured Poets Department,Taylor Swift,Jack Antonoff,"Taylor Swift, Jack Antonoff",12338416
2,Down Bad,Taylor Swift,Jack Antonoff,"Taylor Swift, Jack Antonoff",16605339
3,Is It Over Now? (Taylor’s Version) [From the V...,Taylor Swift,Jack Antonoff,"Taylor Swift, Jack Antonoff",39
4,Cruel Summer,Taylor Swift,Jack Antonoff,"St. Vincent, Jack Antonoff, Taylor Swift",47224140
5,august,Taylor Swift,Jack Antonoff,"Jack Antonoff, Taylor Swift",41021535
6,Guilty as Sin?,Taylor Swift,Jack Antonoff,"Taylor Swift, Jack Antonoff",21127899
7,I Can Do It With a Broken Heart,Taylor Swift,Jack Antonoff,"Taylor Swift, Jack Antonoff",18302081
8,Anti-Hero,Taylor Swift,Jack Antonoff,"Taylor Swift, Jack Antonoff",33296393
9,Look What You Made Me Do,Taylor Swift,Jack Antonoff,"Fred Fairbrass, Jack Antonoff, Richard Fairbra...",22314560
